# ディープ畳み込み敵対的生成ネットワーク（DCGAN）

**学習目標**

1. Keras Sequential API を使用して Generator（生成器）と Discriminator（識別器）モデルを定義する
2. `BinaryCrossentropy` を使用して Generator と Discriminator の損失関数を実装する
3. `tf.GradientTape` を使用して GAN のトレーニングステップを実装し、`tf.function` でコンパイルする
4. `tf.data` データセットパイプラインを使用して GAN をトレーニングし、生成された画像を検査する


## はじめに

このノートブックでは、[Deep Convolutional Generative Adversarial Network](https://arxiv.org/pdf/1511.06434.pdf)（DCGAN）を使用して手書き数字の画像を生成する方法を解説します。コードは、`tf.GradientTape` トレーニングループを備えた [Keras Sequential API](https://www.tensorflow.org/guide/keras) を使用して記述されています。

### GAN とは？
[敵対的生成ネットワーク](https://arxiv.org/abs/1406.2661)（Generative Adversarial Networks: GAN）は、現代のコンピュータサイエンスにおいて最も興味深いアイデアの 1 つです。敵対的プロセスによって 2 つのモデルが同時にトレーニングされます。*Generator*（生成器: 「芸術家」）は本物に見える画像を作成することを学習し、*Discriminator*（識別器: 「美術評論家」）は本物の画像と偽物の画像を判別することを学習します。

![Generator と Discriminator の図](https://tensorflow.org/images/gan/gan1.png)

トレーニング中、*Generator* は本物のような画像を作成する能力が徐々に向上し、*Discriminator* はそれらを見分ける能力が徐々に向上します。このプロセスは、*Discriminator* が本物の画像と偽物の画像を区別できなくなったときに平衡状態に達します。

![トレーニング中の Generator と Discriminator の図](https://tensorflow.org/images/gan/gan2.png)

このノートブックでは、MNIST データセットを用いてこのプロセスを実践します。以下のアニメーションは、50 エポックにわたってトレーニングされた *Generator* によって生成された一連の画像を示しています。画像は最初ランダムなノイズですが、時間が経つにつれて徐々に手書き数字らしくなっていきます。

![出力サンプル](https://tensorflow.org/images/gan/dcgan.gif)


## Keras とその他のライブラリのインポート


In [ ]:
import os
import warnings

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import glob
import os
import time

import imageio
import keras
import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf
from IPython import display
from keras import layers

## データセットの読み込みと準備

このノートブックでは、Generator と Discriminator をトレーニングするために MNIST データセットを使用します。Generator は、MNIST データに類似した手書き数字を生成します。


In [ ]:
(train_images, train_labels), (_, _) = keras.datasets.mnist.load_data()

In [ ]:
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1).astype(
    "float32"
)
train_images = (train_images - 127.5) / 127.5  # 画像を [-1, 1] の範囲に正規化

In [ ]:
BUFFER_SIZE = 60000
BATCH_SIZE = 256

次に、`tf.data` を使用して入力パイプラインを定義します。以下のパイプラインは、`train_images` をテンソルスライスとして読み込み、トレーニング用にサンプルをシャッフルしてバッチ処理します。


In [ ]:
# データをバッチ処理してシャッフル
train_dataset = tf.data.Dataset.from_tensor_slices(train_images)
train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

## Generator と Discriminator モデルの作成

Generator と Discriminator の両モデルとも、[Keras Sequential API](https://keras.io/guides/sequential_model/) を使用して定義します。


### Generator（生成器）

Generator は [keras.layers.Conv2DTranspose](https://keras.io/api/layers/convolution_layers/convolution2d_transpose/)（アップサンプリング / 転置畳み込み）レイヤーを使用して、シード（ランダムノイズ）から画像を生成します。まずこのシードを入力として受け取る `Dense` レイヤーから始め、目標の画像サイズである 28x28x1 に達するまで複数回アップサンプリングを行います。


**演習.** 以下のコードを完成させて、Generator モデルを作成してください。ランダムノイズを入力として受け取る Dense レイヤーから始めます。ランダムノイズは `keras.random.normal([1, 100])` を使用して作成します。複数の `keras.layers.Conv2DTranspose` レイヤーを使用して、100 次元のランダムノイズを最終的な 28x28x1 の次元（元の MNIST 数字の形状）へとアップサンプリングします。

ヒント: `BatchNormalization` の使用や、`LeakyReLU` などのさまざまな活性化関数を試してみてください。


In [ ]:
# TODO 1


def make_generator_model():
    model = keras.Sequential()
    model.add(layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((7, 7, 256)))

    model.add(
        layers.Conv2DTranspose(
            128, (5, 5), strides=(1, 1), padding="same", use_bias=False
        )
    )
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(
        layers.Conv2DTranspose(
            64, (5, 5), strides=(2, 2), padding="same", use_bias=False
        )
    )
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(
        layers.Conv2DTranspose(
            1,
            (5, 5),
            strides=(2, 2),
            padding="same",
            use_bias=False,
            activation="tanh",
        )
    )
    assert model.output_shape == (None, 28, 28, 1)

    return model

まだトレーニングされていない Generator を使用して、画像を作成してみましょう。


In [ ]:
generator = make_generator_model()

noise = keras.random.normal([1, 100])
generated_image = generator(noise, training=False)

plt.imshow(generated_image[0, :, :, 0], cmap="gray")

`.summary()` を使用して、Generator と Discriminator モデルの概要を確認できます。


In [ ]:
generator.summary()

In [ ]:
keras.utils.plot_model(
    generator, show_shapes=True, show_layer_names=True, dpi=75
)

### Discriminator（識別器）

次に、Discriminator を構築します。Discriminator は CNN ベースの画像分類器です。28x28x1 の形状の画像を入力として受け取り、その画像が本物かどうかを示す単一の分類結果を出力します。


**演習.** 以下のコードを完成させて、CNN ベースの Discriminator モデルを作成してください。モデルは、28x28x1 の形状のテンソルを入力として受け取るバイナリ分類器である必要があります。畳み込み層のスタック、活性化関数、Dropout などのさまざまな組み合わせを試してみてください。


In [ ]:
# TODO 1.


def make_discriminator_model():
    model = keras.Sequential()
    model.add(
        layers.Conv2D(
            64, (5, 5), strides=(2, 2), padding="same", input_shape=[28, 28, 1]
        )
    )
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding="same"))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))
    assert model.output_shape == (None, 1)

    return model

まだトレーニングされていない Discriminator を使用して、生成された画像が本物か偽物かを分類してみましょう。モデルは本物の画像に対して正の値を、偽物の画像に対して負の値を出力するようにトレーニングされます。


In [ ]:
discriminator = make_discriminator_model()
decision = discriminator(generated_image)
print(decision)

In [ ]:
discriminator.summary()

In [ ]:
keras.utils.plot_model(
    discriminator, show_shapes=True, show_layer_names=True, dpi=75
)

## 損失関数とオプティマイザの定義

次に、Generator と Discriminator の両モデルの損失関数とオプティマイザを定義します。Generator と Discriminator の両方に `BinaryCrossentropy` を使用します。

Discriminator モデルは（Sigmoid レイヤーからの確率ではなく）ロジットを返すため、`from_logits` を `True` に設定する必要があります。


In [ ]:
# このメソッドは交差エントロピー損失を計算するヘルパー関数を返します
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)

### Discriminator の損失

以下のメソッドは、Discriminator が本物の画像と偽物の画像をどれだけうまく識別できるかを定量化します。

Discriminator をトレーニングする際（つまり Generator を固定した状態）、損失関数は 2 つの部分から構成されます: 本物のデータからサンプリングした場合の損失と、偽物のデータからサンプリングした場合の損失です。以下の関数は、本物の画像に対する Discriminator の予測を 1 の配列と比較し、偽物（生成された）の画像に対する Discriminator の予測を 0 の配列と比較します。


**演習.**
以下のメソッドのコードを完成させてください。`real_loss` は本物の画像に対する Discriminator の予測の交差エントロピーを返し、`fake_loss` は偽物の画像に対する Discriminator の予測の交差エントロピーを返す必要があります。


In [ ]:
# TODO 2


def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(keras.ops.ones_like(real_output), real_output)
    fake_loss = cross_entropy(keras.ops.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

### Generator の損失
Generator の損失は、Discriminator をどれだけうまく欺くことができたかを定量化します。直感的には、Generator が適切に機能している場合、Discriminator は偽物の画像を本物（つまり 1）として分類します。ここでは、生成された画像に対する Discriminator の判定を 1 の配列と比較します。


**演習.**
コードを完成させて、Generator の出力の交差エントロピー損失を返してください。


In [ ]:
# TODO 2


def generator_loss(fake_output):
    return cross_entropy(keras.ops.ones_like(fake_output), fake_output)

### Generator と Discriminator のオプティマイザ

2 つのネットワークを別々にトレーニングするため、Discriminator 用と Generator 用に 2 つの独立したオプティマイザを定義する必要がある点に注意してください。


In [ ]:
generator_optimizer = keras.optimizers.Adam(1e-4)
discriminator_optimizer = keras.optimizers.Adam(1e-4)

### チェックポイントの保存
このノートブックでは、モデルを保存および復元する方法も示します。これは、時間のかかるトレーニングタスクが中断された場合に役立ちます。


In [ ]:
checkpoint_dir = "./gan_training_checkpoints"
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator=generator,
    discriminator=discriminator,
)

## トレーニングループの定義

次に、GAN をトレーニングするためのトレーニングループを定義します。以下でトレーニング用のグローバル変数を設定します。


In [ ]:
EPOCHS = 50
NOISE_DIM = 100
NUM_EXPAMPLES_TO_GENERATE = 16

# このシードを繰り返し再利用します（アニメーション GIF で
# 進行状況を視覚化しやすくするため）
seed = keras.random.normal([NUM_EXPAMPLES_TO_GENERATE, NOISE_DIM])

トレーニングループは、Generator がランダムシードを入力として受け取ることから始まります。そのシードを使用して画像が生成されます。次に、Discriminator を使用して本物の画像（トレーニングセットから取得）と偽物の画像（Generator によって生成）を分類します。これらの各モデルに対して損失が計算され、勾配を使用して Generator と Discriminator が更新されます。


**演習.**
以下のコードを完成させて、GAN のトレーニングループを定義してください。以下の `tf.function` の使用に注目してください。このアノテーションにより、関数 `train_step` が「コンパイル」されます。`train_step` 関数は、入力として画像のバッチを受け取ります。関数の残りの部分では:
- `noise` を入力として `generator` 関数を使用して `generated_images` を作成します
- `images` と `generated_images` に Discriminator モデルを適用して、それぞれ `real_output` と `fake_output` を作成します
- 上記で定義したメソッドを使用して `gen_loss` と `disc_loss` を定義します
- それぞれ `gen_tape` と `disc_tape` を使用して Generator と Discriminator の勾配を計算します

最後に、`.apply_gradients` メソッドを使用して `generator_optimizer` と `discriminator_optimizer` の勾配ステップを実行します。


In [ ]:
# TODO 3


@tf.function
def train_step(images):
    noise = keras.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(
        gen_loss, generator.trainable_variables
    )
    gradients_of_discriminator = disc_tape.gradient(
        disc_loss, discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(gradients_of_generator, generator.trainable_variables)
    )
    discriminator_optimizer.apply_gradients(
        zip(gradients_of_discriminator, discriminator.trainable_variables)
    )

上記の `train_step` 関数を使用して、GAN のトレーニングを定義します。ここで `train` 関数は、引数として `tf.data` データセットとトレーニングのエポック数を受け取ります。


In [ ]:
def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()

        for image_batch in dataset:
            train_step(image_batch)

        # GIF 用の画像を随時生成
        display.clear_output(wait=True)
        generate_and_save_images(generator, epoch + 1, seed)

        # 15 エポックごとにモデルを保存
        if (epoch + 1) % 15 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)

        print(f"Time for epoch {epoch + 1} is {time.time() - start} sec")

    # 最終エポック後に生成
    display.clear_output(wait=True)
    generate_and_save_images(generator, epochs, seed)

**画像の生成と保存**
画像を生成して保存するための小さなヘルパー関数を使用します。


In [ ]:
def generate_and_save_images(model, epoch, test_input):
    # training が False に設定されていることに注目してください。
    # これにより、すべてのレイヤーが推論モード（バッチ正規化など）で実行されます。
    predictions = model(test_input, training=False)

    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap="gray")
        plt.axis("off")

    plt.savefig(f"./gan_images/image_at_epoch_{epoch:04d}.png")
    plt.show()

## モデルのトレーニング
上記で定義した `train()` メソッドを呼び出して、Generator と Discriminator を同時にトレーニングします。GAN のトレーニングは繊細であることに注意してください。Generator と Discriminator が互いを圧倒してしまわないこと（例: 同程度のペースでトレーニングが進むこと）が重要です。

トレーニングの開始時には、生成される画像はランダムなノイズのように見えます。トレーニングが進むにつれて、生成される数字はますます本物らしくなっていきます。約 50 エポック後には、MNIST の数字に類似してきます。これには 1 エポックあたり約 1 〜 2 分かかる場合があります。


In [ ]:
!test -d ./gan_images || mkdir ./gan_images/

In [ ]:
# TODO 4
train(train_dataset, EPOCHS)

## GIF の作成

最後に、トレーニングを通じて生成された画像の変化を示す GIF を作成します。


In [ ]:
# エポック番号を使用して単一の画像を表示


def display_image(epoch_no):
    return PIL.Image.open(f"./gan_images/image_at_epoch_{epoch_no:04d}.png")

In [ ]:
display_image(EPOCHS)

`imageio` を使用して、トレーニング中に保存された画像からアニメーション GIF を作成します。


In [ ]:
anim_file = "dcgan.gif"

with imageio.get_writer(anim_file, mode="I") as writer:
    filenames = glob.glob("./gan_images/image*.png")
    filenames = sorted(filenames)
    last = -1
    for i, filename in enumerate(filenames):
        frame = 2 * (i**0.5)
        if round(frame) > round(last):
            last = frame
        else:
            continue
        image = imageio.imread(filename)
        writer.append_data(image)
    image = imageio.imread(filename)
    writer.append_data(image)

display.Image(filename=anim_file)

## 次のステップ


このチュートリアルでは、GAN を記述してトレーニングするために必要なすべてのコードを示しました。次のステップとして、別のデータセット（例えば [Kaggle で入手可能な](https://www.kaggle.com/jessicali9530/celeba-dataset) 大規模有名人の顔属性（CelebA）データセットなど）で実験してみるのもよいでしょう。GAN の詳細については、[NIPS 2016 Tutorial: Generative Adversarial Networks](https://arxiv.org/abs/1701.00160) を参照することをお勧めします。

Copyright 2020 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License
